f# Page renderer - Confluence Publisher Application

This application produces Confluence content based on an information model. 

1. **Build page tree based on configuration and from information model (SSOT JSON)**
1. **Render content (page, attachments, ...) including translations**
1. Back up existing content (Optional)
1. Upload content

## Configuration

Read and validate configuration.

Be aware not to commit your credentials stored in the configuration files.

In [ ]:
configuration_file = 'geberit.yaml'
configuration_file = 'AGRAVIS.yaml'
configuration_file = 'raetsel-3lang.yaml'
configuration_file = 'crm_model.yaml'
configuration_file = 'AGRAVIS.yaml'
configuration_file = 'modelmodel.yaml'
configuration_file = 'crm_model.yaml'
configuration_file = 'borr.yaml'
configuration_file = 'confluence-cloud-demo.yaml'
#configuration_file = 'bopt.yaml'
configuration_file = 'fyayc-projects.yaml'

In [ ]:
import argparse
import sys
import os
from pathlib import Path
import yaml
import json
import copy


parser = argparse.ArgumentParser(description='Generate SSOT and diagrams from ODM model')
parser.add_argument('configfile', nargs='?',
                    help=f"Path of the configuration (yaml) file for confluence rendering.")
parser.add_argument('jsonfile', nargs='?',
                    help=f"Path jsonf file to be confluence rendered.")

arguments = argparse.Namespace()

if len(sys.argv) > 0 and '.py' in sys.argv[0] and not 'ipykernel' in sys.argv[0]:
    arguments = parser.parse_args()
    if arguments.configfile is not None:
        configuration_file = arguments.configfile
    if arguments.jsonfile is not None:
        json_ssot = arguments.jsonfile
else:
    # in jupyter environment
    assert 'ipykernel' in sys.argv[0], f"Expecting to run in Jupyter environment 🙀"

assert os.path.isfile(configuration_file), "Cannot read file " + configuration_file
print(f"Working with configuration from {configuration_file}")


In [ ]:
## Read configuration values
with open(configuration_file) as f:
    config = yaml.safe_load(f)
assert len(config) > 0, f'Config is empty :-('

if json_ssot is None:
    #old solution with json in configfile
    json_ssot = config['json']
assert os.path.isfile(json_ssot), f'Source file {json_ssot} not found'

database_directory = Path(json_ssot).parent

source_directory = database_directory.parent

log_directory = source_directory / 'log'
os.makedirs(log_directory,exist_ok=True)

content_root = source_directory / 'confluence-content'
os.makedirs(content_root, exist_ok=True)
print(f"Publishing to {os.path.abspath(content_root)}")

In [ ]:
## Pathes
curfile=__file__
if os.path.dirname(curfile).endswith("dist"):
    #python file, go one directory  up
    CODEROOT = Path(curfile).parent.parent
elif os.path.dirname(curfile).endswith('confluence-export'):
    #jupyter notebook file, go 2 directorys up up
    CODEROOT = Path(curfile).parent.parent.parent
else:
    assert False,curfile

IM_TOOL_LIB = CODEROOT / 'pythonWork'/'pythonSource'
assert IM_TOOL_LIB.is_dir(), f"{IM_TOOL_LIB.resolve()} is not a folder"
sys.path.insert(0, str(IM_TOOL_LIB.resolve()))

RENDER_TOOL_LIB = CODEROOT / "notebooks" / "confluence-export"
LOCALLANGS = RENDER_TOOL_LIB / "locale"
JINJATEMPLATES = RENDER_TOOL_LIB / "templates"
TESTDATA = RENDER_TOOL_LIB / "testdata"



## Logging

In [ ]:


import logging
from logging import handlers
from datetime import datetime

logfile = log_directory / 'renderer.log'

formatter = logging.Formatter("%(asctime)s [%(threadName)s] - %(name)s - %(levelname)s - %(message)s")
file_log_handler = handlers.RotatingFileHandler(logfile, maxBytes=(1024 * 1024 * 10), backupCount=10)
file_log_handler.setFormatter(formatter)
file_log_handler.setLevel(logging.DEBUG)

console_formatter = logging.Formatter("%(relativeCreated)5d %(levelname)s - %(message)s")
console_log_handler = logging.StreamHandler()
console_log_handler.setFormatter(console_formatter)
console_log_handler.setLevel(logging.INFO)

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
logger.addHandler(file_log_handler)
logger.addHandler(console_log_handler)

logger.info(f"Logging to {logfile} initialised")

In [ ]:

print(f"Loading {json_ssot}")
with open(json_ssot, 'r', encoding='utf-8') as source:
     data = json.load(source)

In [ ]:
data['model']['name'], data['_imprint_']

In [ ]:
def sanitize_filename(name: str) -> str:
    return "".join(c for c in name if c.isalnum() or c in ('.', '-', '_', ' ')).rstrip()

In [ ]:
# sanitize_filename(data['model']['name']


## Evaluate if we are online (deprecated -> two scripts)

In [ ]:
confluence = config.get('confluence')
if confluence is not None:
    destination = f"{config['confluence']['apiurl']}/{config['confluence']['space']}/{config['confluence']['rootpage']}"
else:
    destination = f" folder {os.path.abspath(content_root)} only "

In [ ]:
from IPython.core.display import HTML
HTML('''<p><span style="font-family: Impact; font-size:48px">
Publishing the information model <span style="color: darkorange">{name}
</span></span><br/> 
to <span style="color: darkorange">{dest}</span></p><p>from {source} according to configuration from {config}</p>'''
   .format(name=data['model']['name'], source=json_ssot,
           config=os.path.abspath(configuration_file), dest=destination))
     

In [ ]:
if confluence is not None:
    assert len(confluence['apiurl']) > 0
    space_key = confluence['space']
    root_page = confluence['rootpage']

    conf_confidential = copy.deepcopy(config)
    conf_confidential['confluence']['password'] = '***'
    print(f"Exporting to confluence {conf_confidential}")

## Verify dependencies
Do this early before processing anything

In [ ]:
import glob
import sys
from lxml import etree


from SSOT_db.IM_OBJECTS.modelelement import Modelelemtype
from SSOT_db.IM_JSON import JSModel,jsguid2type

from IM_WEB.IM_HTML import drawiodiagram
from IM_WEB.IM_HTML import entityenviron

## Initialize logging

In [ ]:
import logging

log = logging.getLogger()
log.setLevel(logging.DEBUG)

LOGFILE = log_directory / 'debug.log'
#os.makedirs('target', exist_ok=True)

handler = logging.handlers.RotatingFileHandler(
    LOGFILE, maxBytes=(1048576*5), backupCount=7
)
formatter = logging.Formatter("%(asctime)s [%(threadName)s] - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)
handler.setLevel(logging.DEBUG)
log.addHandler(handler)

log.debug('Starting execution')

# Setup translation

Requires gettext: `conda install gettext`

Trigger scan for translatable objects:
    `xgettext --from-code utf-8 -L python -d scan.pot templates/*`
    `msgmerge --width --update

https://phrase.com/blog/posts/translate-python-gnu-gettext/

## Internationalisation (i18n)

Publishing of content in different languages is supported. The GNU `gettext` toolchain is used to translate texts in code and templates.

Translation files are located in the 'locale' directory structure.

Use `xgettext --from-code utf-8 -L python -d confluence-publisher templates/*` to scan for content and
`msgfmt -o ../locale/de/LC_MESSAGES/confluence-publisher.mo ../locale/de/LC_MESSAGES/confluence-publisher.po` to upate the compiled translation files.

Implemtation in section 'Translation'

In [ ]:
import subprocess
import glob
from pathlib import Path

try:
    result = subprocess.run("msgfmt -V", shell=True, check=True, capture_output=True)
    message = result.stdout.decode()
    assert message.find('msgfmt') >= 0, 'gettext tool msgfmt is missing. Install it using `brew install gettext`.\n-= Message =-\n' + str(message)
    
    sources = glob.glob(os.path.abspath(LOCALLANGS)+'/**/*.po', recursive=True)
    try:
        for src in sources:
            source = Path(src)
            destination = source.with_suffix('.mo')
            if source.stat().st_mtime > destination.stat().st_mtime: 
                log.info("Updating compiled translation {0} from {1}".format(destination, source))
                subprocess.run('msgfmt -o {dest} {src}'.format(dest=str(destination), src=str(source)), shell=True, check=True, capture_output=False)
                print("Translation {0} updated".format(src))
    except subprocess.CalledProcessError as e:
        raise Exception('Cannot update translation', e)
except subprocess.CalledProcessError as e:
    print(e)
    log.warning('Cannot update translation inline. But this is ok')
    pass

In [ ]:
import gettext
locale_folder = config.get('locale', LOCALLANGS)
gettext.bindtextdomain('confluence-publisher', locale_folder)

In [ ]:
from pathlib import Path
translations_folders = list(Path(locale_folder).rglob("LC_MESSAGES"))
translations_folders

## Translator

Translation is done using the custom translator
It uses gettext translate when there is no translation provided in the SSOT.



In [ ]:
class Translator:
    """Translate strings"""
    logger = logging.getLogger("Translator")
    
    def __init__(self, language: str):
        self.language = language
        self.translator = gettext.translation('confluence-publisher',
                                              locale_folder, fallback=True, languages=[language])
        self.title_format = '{title} - [{language}]'
        self.logger = logging.getLogger("Translator " + language)
        
    def tr(self, element) -> str:

        if isinstance(element, dict):
            """If the value provided is a field containing translations, use them"""
            text = element.get(self.language)
            if text is None: #and len(element.values()) > 0:
                text = list(element.values())[0]
                self.logger.warning('Translator: Falling back to {} from {}'.format(text, str(element)))
            if not text:
                return ''
            return text

        # Fallback to gettext if not a dict
        if isinstance(element, str):
            translated = self.translator.gettext(element)
            return translated

        self.logger.warning("Cannot translate element '{0}' of type {1}".format(element, type(element)))
        return None
    
    def gettext(self, text: str):
        result = self.translator.gettext(text)
        if result == text:
            self.logger.warning("No translation for {0}".format(text))
        return result
    
    def translator(self):
        return self.translator
    
    def title_language(self, title: str) -> str:
        """Create unique confluence page title per translation"""
        return self.title_format.format(title = title, language = self.language)
    
    def key_lang(self, key: str) -> str:
        return key + '-' + self.language
    
    def lang(self) -> str:
        return self.language

In [ ]:
translators = { language: Translator(language) for language in config['languages'] }
translators

In [ ]:
default_language_translator = translators[config['languages'][0]]
default_language_translator.title_format = '{title}'
'Default language is {}'.format(default_language_translator.lang())

### Selftests

In [ ]:
result = default_language_translator.tr('fadsjfklj4q9u')
assert result == 'fadsjfklj4q9u'

In [ ]:
default_language_translator.tr('Synonyms')

In [ ]:
default_language_translator.tr({ 'es': 'hola'} )

In [ ]:
default_language_translator.tr('Diagram')

In [ ]:
default_language_translator.gettext('Entities')

In [ ]:
default_language_translator.gettext('{parent_title} - Documentation')

## Load the data
The **data** is the JSON serialized information model 

In [ ]:
import json

data = None
with open(json_ssot, 'r') as source:
     data = json.load(source)

str(data)[:512]

In [ ]:
data.get('_imprint_')

In [ ]:
model_languages = list(data['languages'])
config_languages = list(config['languages'])
'Model languages: {}. Export configuration selected languages: {}'.format(model_languages, config_languages)

In [ ]:
# Missing languages?
missing_translations = set(config_languages) - set(model_languages)
assert len(missing_translations) == 0, 'Missing languages in model: {}'.format(missing_translations) 

In [ ]:
categories = list(data)
for category in categories:
    print('{} Elements in category "{}"'.format(len(data[category]), category))

# Setup structure (page hierarchy) and navigation util

In [ ]:
disclaimer = config.get('disclaimer', '')

destination_folder = 'pages'

## Navigator
The navigator contains the map of all pages. It allows create confluence links from one page to it's translations and to parent pages.
The internal page map contains:
'key': element key without language suffix -> 'page': Confluence page objects

In [ ]:
class Navigator:
    
    def __init__(self):
        self.page_map = {}
        self.page_uniqueness_map = {}
    
    def register_page(self, key: str, lang: str, page: dict):
        key_lang = self.title_lang(key, lang)
        previous = self.page_map.get(key_lang)
        assert (not previous) or previous == page, "There is already a page with key {0}: {1}-----------{2}".format(key_lang, previous, page)
        self.page_map[key_lang] = page
        
        page['id'] = key_lang
        
        title = page['title']
        previous_u = self.page_uniqueness_map.get(title)
        self.page_uniqueness_map[title] = page
        assert not previous_u, "There is already a page with title {0}: {1}------------{2}".format(title, previous_u, page)
    
    def title_lang(self, key: str, lang: str) -> str:
        return key + '-' + lang
    
    def page_4_language(self, key: str, lang: str) -> str:
        return self.page_map.get(self.title_lang(key, lang))
    
    def translation_title(self, key: str, lang: str) -> str:
        """Get the title of the page with 'key' for language 'lang'"""
        return self.translation_page(key, lang)['title']

    def pages(self) -> list:
        return list(self.page_map.values())
    
    def page_by_title(self, title: str) -> dict:
        return self.page_uniqueness_map.get(title)  
    
    def page_by_key(self, key: str, lang: str) -> dict:
        return self.page_map[self.title_lang(key, lang)]

## Prepare destination structure
Configuration:
- content
  - Entities
    - Attributes
  - Databases
    - Tables
      - Columns

Rolled out:
- Topic Entities
  - Entity First
      - Attribute1 of first entity
      - Attribute2 of first entity
  - Entity Second
      - Attribute1 of second entity
- Topic 'Databases'
  - System A
    - Table A1
      - Column ID - A1
      - Colunn Name - A1
      - Column Value - A1
    - Table A2
  - System B
    - Table B1
    - Table B2

In [ ]:
def ensure_unique_page_title(page: dict, navigator: Navigator):
    """Make sure the page title is unique in this Confluence space. This does eventually modify the page element!"""
    title = page['title']
    existing_page_with_same_title = navigator.page_by_title(title)
    if existing_page_with_same_title:
        #print('Page for {}[{}] has same title as {}[{}]\nOld:{}\nNew:{}'.format(
        #    existing_page_with_same_title['key'], existing_page_with_same_title['title'], page['key'], title,
        #    existing_page_with_same_title['item'], page['item']))
        tokens = title.split('-')
        if len(tokens) > 1:
            tokens.insert(len(tokens) - 1, page['key'])
        else:
            tokens.append(page['key'])
        page['title'] = ' - '.join(tokens)
        print('Created unique title {}'.format(page['title']))


def manual_documentation(parent_page: dict, config: dict, navigator: Navigator, translator: Translator):
    """Add a page for manual documentation, if configured"""
    md = config.get('manual-documentation')
    if md:
        page = copy.copy(parent_page)
        page_key = parent_page['key'] + '-manual-documentation'
        page['key'] = page_key
        title_format = translator.gettext(md.get('title-format', "{parent_title} - manual"))
        page['name'] = title_format.format(parent_name=parent_page['name'], key=parent_page['key'], lang=translator.lang(), parent_title=parent_page['title'])
        page['title'] = page['name']
        page['config'] = md
        page['parent'] = parent_page
        page['template'] = translator.gettext(md.get('template', "manual-documentation.templ.html")) #'Manual documentation for {title} [{key}]'.format(title=parent_page['title'], key=page['key'])))
        # register child with parent
        parent_page['manual-documentation-page-title'] = page['title']
        
        labels = set(md.get('labels', []))
        labels.add('manual')
        labels.add(translator.lang())
        page['labels'] = labels
        # keep confluence content
        page['preserve'] = True
        page['level'] = page['level'] + 1
        page['template'] = md.get('template')
        page['path'] = parent_page['path']
        page['file'] = f"{parent_page['key']}-{translator.lang()}-manual.confluence.xml"
        
        ensure_unique_page_title(page, navigator)
        navigator.register_page(page_key, translator.lang(), page)
        return page
    else:
        return None

def sortdocuments(docus:dict)->list:
    """ sort documents by level.
        first no parent, then parentlevel 1 then parentlevel...
        """
    docukeys=[key for key,docu in docus.items() if docu['parent'] is None]
    while len(docukeys) < len(docus.keys()):
        for key,docu in docus.items():
            if key in docukeys: continue
            if docu['parent'] in docukeys:
                docukeys.append(key)
    return docukeys


def prepare_pages(parent_page: dict, config: dict, topic: str, level: int, navigator: Navigator, translator: Translator, data: object):
    elements = data[topic]
    pages = []
    item_filter = config.get('filter')
    filtered = 0
    keylist = list(elements) if topic != 'documents' else sortdocuments(elements)
    for element_key in keylist:

        item = data[topic][element_key]

        if item_filter:
            filter_result = eval(item_filter)
            if not filter_result:
                log.debug(f"Hiding page {element_key} due to filter result")
                filtered += 1
                continue

        name_translated = translator.tr(item['name']).strip()
        if type(name_translated) != str and len(name_translated) > 1:
            log.warning("Strange name for item {0}".format(item))

        title_safe = name_translated
        
        if topic == 'attributes':
            parent_key = item['entity']
        elif topic == 'tables':
            parent_key = item['datamodel-id']
        elif topic == 'columns':
            parent_key = item['table-id']
        #elif topic == 'documents' and item['parent'] is not None:
        #    parent_key = item['parent']
        else:
            parent_key = parent_page['key']
        
        parent = navigator.page_4_language(parent_key, translator.lang())
        if parent is None:
            log.debug(f"Hiding page {element_key} due to suppressed parent {parent_key}")
            filtered += 1
            continue
        
        assert parent, f"Missing parent page {parent_key} for language {translator.lang()}"
        
        parent['child-count'] = parent['child-count'] + 1
        parent_name = parent['name']
        
        item_title_rule = config.get('title_rule')
        if item_title_rule:
            title_by_rule = eval(translator.gettext(item_title_rule))
            if title_by_rule:
                title_safe = title_by_rule
        else:
            # Default naming rule for nested elements: {child_title} - {parent_title}
            title_format = translator.gettext(config.get('title-format', "{child_title} - {parent_name}"))
            if topic in ['attributes', 'tables', 'columns']:
                title_safe = translator.gettext(title_format).format(child_title=name_translated, parent_name=parent_name,
                                                                     parent_title=parent['title'], lang=translator.lang())
                                              
        title = title_safe
        
        labels = set(config.get('labels', []))
        labels.add(translator.lang())
        page = {
            'key': element_key,
            'topic': topic,
            'name': title,
            'title': translator.title_language(title),
            'parent': parent,
            'labels': list(labels),
            'item': item,
            'config': config,
            'level': level,
            'translator': translator,
            'template': translator.gettext(config.get('template')),
            'child-count': 0,
            'path': os.path.join(destination_folder, topic),
            'file': f"{element_key}-{translator.lang()}.confluence.xml"
        }
        
        ensure_unique_page_title(page, navigator)
        
        navigator.register_page(element_key, translator.lang(), page)
        pages.append(page)
        
        if len(pages) < 2:
            print("Prepared page '{0}' [{1}]".format(page['title'], page['key']))
                
        md = manual_documentation(page, config, navigator, translator)
        if md:
            pages.append(md)
            if len(pages) < 3:
                print("Prepared manual documentation page '{0}' [{1}]".format(md['title'], md['key']))

        print('Added {} pages for topic {}. Filtered out {}'.format(len(pages), topic, filtered))

    # Descend into children, if any ...
    child_configurations = config.get('content')
    if child_configurations:
        """Process child types"""
        for child_config_topic in child_configurations:
            if not data[child_config_topic]:
                log.warning('No data for topic "{}"'.format(child_config_topic))
                continue
            child_config = child_configurations[child_config_topic]
            log.info(f"Processing {child_config_topic} on level {level}")
            subpages = prepare_pages(parent_page=None, config=child_config, topic=child_config_topic,
                                     level=level + 1, navigator=navigator, translator=translator, data=data)

    return pages


def top_level_content(config: dict, navigator: Navigator, translator: Translator, data: dict, base_position: int) -> list:
    """Recurse configuration content structure"""
    content = config.get('content')
    pages = []
    if content:
        position = base_position
        for topic_key in list(content):
            topic_config = content[topic_key]
            labels = set(topic_config.get('labels', []))
            labels.add('im-parent')
            labels.add('im-parent-' + topic_key)            
            labels.add(translator.lang())
            name = translator.gettext(topic_config.get('title')) #+ ' - ' + data['model']['name']
            category_page = {
                'topic': topic_key + '-root',
                'key': topic_key,
                'name': name,
                'title': translator.title_language(name),
                'labels': list(labels),
                'config': topic_config,
                'parent': None,
                'level': 0,
                'position': position,
                'child-count': 0,
                'translator': translator,
                'template': translator.gettext(topic_config.get('index-template', 'parent-page-index.templ.html')),
                'path': os.path.join(destination_folder, topic_key),
                'file': f"{topic_key}-{translator.lang()}.confluence.xml",
            }
            
            ensure_unique_page_title(category_page, navigator)

            navigator.register_page(topic_key, translator.lang(), category_page)
            pages.append(category_page)

            if len(pages) < 2:
                print("Prepared top level page '{0}' [{1}]".format(category_page['title'], category_page['key']))

            print('Processing category {} [{}]'.format(topic_config.get('title'), topic_key))
                        
            sub_pages = prepare_pages(category_page, topic_config, topic_key, 1, navigator, translator, data)
            if len(sub_pages) == 0:
                category_page['skip'] = True
            position += 1
    else:
        log.error('Cannot find content on root level')
    return pages

In [ ]:
navigator = Navigator()

base_position = 0
for lang in list(config['languages']):
    print('*** Scanning for language {0} ***'.format(lang))
    trans = translators[lang]
    print('Translation => tr:{0} gettext:{1}'.format(trans.tr('Entities'), trans.gettext('Entities')))
    pages = top_level_content(config, navigator, trans, data, base_position)
    print('---------------------------')
    base_position += 10000
    
total = len(navigator.pages())
'Will produce {} * {} ~= {} pages'.format(len(config['languages']), len(pages), total)

In [ ]:
level0 = list(filter(lambda page: page['level'] == 0, navigator.pages()))
[ page.get('title') for page in level0 ]

In [ ]:
skiplist = list(filter(lambda page: page.get('skip', False), navigator.pages()))
'Will skip {} pages: {} ...'.format(len(skiplist), [page.get('title') for page in skiplist[:5]])

# Create Confluence content
Render the page content and create graphs and attachments for the respective page.

In [ ]:
from tqdm.autonotebook import tqdm
from tqdm.notebook import tqdm_notebook

## Helper class to simplify template rendering

This helper is available in jina2 templates with the name 'util'

In [ ]:
import html
import markupsafe
from functools import reduce

class ConfluenceContentUtil:
    """This util is used in jinja2 scripts to create Confluence content.
    It is designed to provide complex functionality, that does not fit into templates directly."""
    def __init__(self, navigator: Navigator, translator: Translator, languages: list, json_data: dict):
        self.translator = translator
        self.navigator = navigator
        self.language = translator.language
        assert len(self.language) == 2
        self.other_lang = list(languages)
        self.other_lang.remove(self.language)
        assert len(self.other_lang) + 1 == len(languages)
        self.json_data = json_data
    
    def translate(self, text):
        return self.translator.tr(text)
    
    def translate_text(self, field) -> markupsafe.Markup:
        translated = self.translator.tr(field)
        if translated is None:
            translated = ''
        text = html.escape(translated)
        linebreaks = text.replace('\n', '<br/>\n')
        return markupsafe.Markup(linebreaks)
        
    def other_languages(self) -> list:
        return self.other_lang
    
    def soft_link(self, key: str, lang: str = None) -> markupsafe.Markup:
        """Returns a confluence link if the key is represented with a page in the same language context or the language provided"""
        if key:
            language = lang if lang else self.language
            page = self.navigator.page_4_language(key, language)
            if page:
                return markupsafe.Markup('''<ac:link><ri:page ri:content-title="{page_title}"/><ac:plain-text-link-body><![CDATA[{name}]]></ac:plain-text-link-body></ac:link>'''.format(
                    page_title=html.escape(page['title']), name=html.escape(page.get('name'))))
            else:
                if 'DOMA' in key:
                    return translator.tr(self.json_data['domains'][key]['name'])
                elif 'COLU' in key:
                    return translator.tr(self.json_data['columns'][key]['name'])
                else:
                    return key
        else:
            return ''
        
    def relation_self(self, entity_key: str, relation_key: str):
        """Returns the local end of the relation_key attached to enitity_key"""
        relation = self.json_data['relations'][relation_key]
        if relation['from-to']['enti'] == entity_key:
            return relation['from-to']
        else:
            return relation['to-from']

    def relation_other(self, entity_key: str, relation_key: str):
        """Returns the remote end of the relation_key"""
        relation = self.json_data['relations'][relation_key]
        if relation['from-to']['enti'] == entity_key:
            return relation['to-from']
        else:
            return relation['from-to']
        
    def column_lineage(self, column_key: str):
        """Collects columns that are mapped with the column provided via the IM"""
        column = self.json_data['columns'][column_key]
        result = []
        for attribute_key in column['attributesmapped']:
            attribute = self.json_data['attributes'][attribute_key]
            columns_mapped = attribute['columnsmapped+']
            all_columns = map(lambda entry: columns_mapped[entry], columns_mapped)
            cols = reduce(lambda e, l: e + l, list(all_columns), [])
            result.extend(cols)
        try:
            result.remove(column_key)
        except ValueError:
            # fine if it is not in the list
            pass
        return result
    
    def attribute_lineage(self, attribute_key: str):
        """Collects columns that are mapped to the provided attribute"""
        attribute = self.json_data['attributes'][attribute_key]
        columns_mapped = attribute['columnsmapped+']
        all_columns = map(lambda entry: columns_mapped[entry], columns_mapped)
        cols = reduce(lambda e, l: e + l, list(all_columns), [])
        return cols
    
    def icon(self, key: str) -> markupsafe.Markup:
        return markupsafe.Markup(
            '<img width="50px" align="right" ' +
            'src="http://res.cloudinary.com/foryouandyourcustomers/image/upload/fyayc_icon_library/svg/0099.svg" />'
        )
        
    def type_name_from_key(self, key: str) -> str:
        """Return the type of a key. Results are [Entity, Attribute, Document, Organisational Unit, Diagram, ...]"""
        if key.startswith(Modelelemtype.ENTI):
            return "Entity"
        elif key.startswith(Modelelemtype.BURU):
            return "Business rule"
        elif key.startswith(Modelelemtype.ATTR):
            return "Attribute"
        elif key.startswith(Modelelemtype.RELA):
            return "Relation"
        elif key.startswith(Modelelemtype.DATM):
            return "Datamodel"
        elif key.startswith(Modelelemtype.DOMA):
            return "Domain"
        elif key.startswith(Modelelemtype.TABL):
            return "Table"
        elif key.startswith(Modelelemtype.DOCU):
            return "Document"
        elif key.startswith(Modelelemtype.COLU):
            return "Column"
        elif key.startswith(Modelelemtype.ORGU):
            return "Organisational unit"
        elif key.startswith(Modelelemtype.DIAG):
            return "Diagram"
        elif key.startswith(Modelelemtype.UDPR):
            return "User defined property"
        elif key.startswith(Modelelemtype.PHYU):
            return "Physical unit"
        elif key.startswith(Modelelemtype.DATY):
            return "Datatype"
        return "Unknown"

    def getelem(self, id):
        return self.json_data[JSModel.elemtype2label(jsguid2type(id))][id]

In [ ]:
from jinja2 import Environment, FileSystemLoader, select_autoescape

def create_jinja2_i18n_env(lang:str) -> Environment:
    jinja_env = Environment(
        loader=FileSystemLoader(JINJATEMPLATES),
        autoescape=select_autoescape(['html', 'xml']),
        extensions=["jinja2.ext.i18n"])
    
    translator = translators[lang]
    jinja_env.install_gettext_translations(translator.translator, newstyle=True)    
    util = ConfluenceContentUtil(navigator, translator, config['languages'], data)
    
    jinja_env.globals.update({ 'util': util, 'disclaimer': disclaimer, 'jinja_env': jinja_env, 'i18n': util })
    return jinja_env

i18n_environments = { lang: create_jinja2_i18n_env(lang) for lang in config['languages'] }

## Graph rendering

In [ ]:
jsmodel = JSModel.readfromfile(pfilename=json_ssot)

In [ ]:
#withtqdm = False

def create_graphs(page: dict) -> str:
    if page.get('topic') == 'entities':
        translator = pmodellang=page['translator']
        env = entityenviron.createentienvironment(pentiid=page['key'], pjson=jsmodel, pmodellang=translator.lang())
        content = entityenviron.generate_drawio_content(penviron=env)
        
        name = f"env-{translator.key_lang(page['key'])}"
        graph_filename = f"{name}.drawio"
        
        graph_folder = os.path.join(content_root, page['path'])
        os.makedirs(graph_folder, exist_ok=True)
        
        file = os.path.join(graph_folder, graph_filename)
        #if withtqdm: pbar.set_description(f"Generating entity grap for {page['id']} [{lang}] to {file}")
        with open(file, 'w') as out:
            out.write(content)
            
        page['entity-graph-name'] = graph_filename
        page['entity-graph-path'] = graph_folder
                                                   
        attachments = page.get('attachments', [])
        diag = { 
            'path': page['path'], 
            'file': graph_filename, 
            'name': name,
            'content-type': 'application/vnd.jgraph.mxfile',
            'labels': ['drawio'],
        }
        attachments.append(diag)
        page['attachments'] = attachments
        page['diagram'] = diag
        return file

## Create diagrams
And the page holding the diagram.

# TODO Create diagram overview page

In [ ]:
import shutil
pagesdirec=os.path.join(content_root,destination_folder)
print (f"remove old directory {pagesdirec} of generated files")
shutil.rmtree(pagesdirec,ignore_errors=True)

if True:
    #with tqdm_notebook(total=len(data['diagrams'].keys())*len(translators), dynamic_ncols=True, unit='Diagram') as pbar:
    #try: #avoid error in gitHub environment (without display)
    #    d = pbar.disp
    #    withtqdm = True
    #except:
    #    withtqdm = False
    for lang, translator in translators.items():
        for key, diagram in data['diagrams'].items():
            path = os.path.join(destination_folder, 'diagrams')            
            folder = os.path.join(content_root, path)
            os.makedirs(folder, exist_ok=True)

            filename = f"{key}-{sanitize_filename(diagram['name'])}-{lang}.drawio"
            file = os.path.join(folder, filename)
            
            #if withtqdm: pbar.set_description(f"Generating diagram {key} '{diagram['name']}' [{lang}] to {file}")
            draw_io_xml = drawiodiagram.create_diagram(key, jsmodel, translator)
            with open(file, 'wb') as out:
                out.write(etree.tostring(draw_io_xml))

            page = navigator.page_by_key(key, lang)
            attachments = page.get('attachments', [])
            diag = { 
                'path': path, 
                'file': filename, 
                'name': f"{key}-{lang}",
                'content-type': 'application/vnd.jgraph.mxfile',
                'labels': ['drawio'],
            }
            attachments.append(diag)
            page['attachments'] = attachments
            page['diagram'] = diag
            
            #if withtqdm: pbar.update(1)

# Generate content

In [ ]:
import re

def write_page_to_disk(page, content: str):
    folder = os.path.join(content_root, page['path'])
    os.makedirs(folder,  exist_ok=True)
    target_file = os.path.join(folder, page['file'])
    with open(target_file, 'w') as out:
        out.write(content)
    page['content'] = content
    return target_file

pages = list(navigator.pages())
if True:
    #with tqdm_notebook(total=len(pages), dynamic_ncols=True, unit='Page') as pbar:
    for page in pages:
        element_config = page['config']
        
        # todo generate graph here
        create_graphs(page)
        
        template_file_name = page.get('template')
        if template_file_name:
            translator = page['translator']
            jinja_env = i18n_environments[translator.language]
            jinja_template = jinja_env.get_template(template_file_name)
            
            rendered = jinja_template.render(page=page, data=data, key=page.get('key'), item=page.get('item'), 
                config=element_config, update_message = 'update')
            
            content = re.sub('<!--.*?->(\n)*', '', rendered) # strip comment lines
            ondisk = write_page_to_disk(page, content)
            page['filename'] = ondisk

        #if withtqdm: pbar.update(1)

## Prepare tasks for simple upload
```
task = {

'url': '/content/{page_id}',
'data': {
    'title': page['title'],
    'id': page['id'],
    'type': page['type'],
    'version': { 'number': 1, 'minorEdit': 'true', 'message': change_message },
    'body': {
        'storage': {
            'representation': 'storage',
            'value': confluence_xml_content_from_file
        }
    }
    'metadata': { 'labels': ['my', 'label'] }
}
}

or in case of attachments
task = { 'url': '/content/{page_id}/child/attachment',
'data': { 'type': 'attachment',
               'fileName': name,
               'contentType': 'application/octet-stream',
               'metadata': { 'labels': ['my', 'label'] },
               'minorEdit': 'true',
               'comment': 'New attachment',
             }

```

create page with:
```
requests.put(url, data=json.dumps(data), headers={'Content-Type': 'application/json'}, auth=auth, proxies = proxies, verify=verify)
```

### Structure to collect publishing tasks 
List of all tasks to be performed to upload content.
Steps have to be executet in sequence, starting at 0.
Tasks in the same step can be executed in parallel.

In [ ]:
steprange = range(0,8)
publish_tasklists = [ {'step': i, 'tasks': [] } for i in steprange ]
publish_tasklists[0]['first'] = 'This is the first step'
publish_tasklists[-1]['last'] = 'This is the last step'
publish_tasklists

In [ ]:
print('Writing confluence content to disk: {}'.format(destination_folder))
os.makedirs(destination_folder, exist_ok=True)

### Tasks
Currently on level 0-4 there will be tasks
On the final level (7), there are attachments

In [ ]:
for level in steprange:
    level_tasklist = publish_tasklists[level]
    level_pages = list(filter(lambda page: page['level'] == level, navigator.pages()))
    print(f"Adding {len(level_pages)} pages on level {level}")
    for page in level_pages:
        
        page_labels = set(page.get('labels', []))
        page_labels.add('generated')
        label_list = list(map(lambda label_name: { 'prefix': 'global', 'name': label_name }, page_labels))
        
        task = { 'url': '/content',
                 'data': { 'title': page['title'],
                    'type': 'page',
                    #'id': page.get('id', None),
                    'space': { 'key': '{space_key}' },
                    #'version': { 'number': 1, 'minorEdit': 'true', 'message': 'Publish' },
                    'body': {
                        'storage': {
                            'representation': 'storage',
                            'value': None,
                        }
                    },
                    'metadata': { 'labels': label_list },
                },
                'source': page['id'],
                'sourcefile': str(os.path.join(page['path'], page['file'])),  # content to body.storage.value
        }
        
        parent_page = page.get('parent')
        if parent_page is not None:
            task['parent'] = parent_page['id']

        task['data']['ancestors'] = [ { 'type': 'page', 'id': '{parent_page_id}' } ]
            
        level_tasklist['tasks'].append(task)
    
    # add all attachments to level 7 
    if level == steprange[-1]:
        pages_with_attachments = list(filter(lambda page: page.get('attachments') is not None, navigator.pages()))
        print(f"Adding attachments of {len(attachments)} pages on final level {level}")
        for page in pages_with_attachments:
            for attachment in page['attachments']:
                
                attachment_labels = set(attachment.get('labels', []))
                attachment_labels.add('generated')
                label_list = list(map(lambda label_name: { 'prefix': 'global', 'name': label_name }, list(attachment_labels)))
 
                task = {
                    'url': '/content/{page_id}/child/attachment',
                    'data': { 'type': 'attachment',
                            'fileName': attachment['file'],
                            'contentType': attachment['content-type'],
                            'minorEdit': 'true',
                            'comment': 'New attachment',
                            'metadata': { 'labels': label_list },
                            },
                    'source': page['id'],
                    'sourcefile': str(os.path.join(attachment['path'], attachment['file'])),  # content to body.storage.value
                }
                level_tasklist['tasks'].append(task)
            

In [ ]:
tasklist_file = os.path.join(content_root, 'tasklist.json')
with open(tasklist_file, 'w') as out:
    json.dump(publish_tasklists, out, indent=2)
print (f"tasklist written to {tasklist_file}")

In [ ]:
from IPython.core.display import HTML
model_name = data['model']['name']
HTML(f"<h1 style=\"color:green;\">Completed generation of {model_name}</h1>")

# Create backup archive

In [ ]:
import zipfile

def zipdir(path, ziph,skipfiles=[]):
    # ziph is zipfile handle
    for root, dirs, files in os.walk(path):
        for file in files:
            if file in skipfiles: continue
            path = os.path.join(root, file)
            ziph.write(path, os.path.relpath(path, content_root))

archivefile=model_name + '.zip'
archive = os.path.join(content_root, archivefile)
zipf = zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED)
zipdir(path=content_root, ziph=zipf,skipfiles=[archivefile])
zipf.close()

print(f"Successfully packed up {archive} from {content_root}")

In [ ]:
pagelist_copy = copy.deepcopy(navigator.page_map)
for page_key in pagelist_copy:
    page = pagelist_copy[page_key]
    page.pop('translator', None)
    stamp = page.get('_publishable-stamp')
    if stamp:
        page['_publishable-stamp'] = stamp.strftime("%Y-%m-%d %H:%M:%S")
    labels = page.get('labels')
    if labels:
        page['labels'] = list(labels)

#ssot = Path('.', 'noindex', model_name + '-pages.json')
ssot = Path(content_root, 'noindex', model_name + '-pages.json')
ssot.parent.mkdir(exist_ok=True)
with open(ssot, 'w') as summary:
    json.dump(pagelist_copy, summary)

## Build code distribution bundle based on the loaded Python modules

In [ ]:
#cwd = Path('.')
cwd = Path(content_root)
project_base = Path(*cwd.absolute().parts[0:-2])
project_base

In [ ]:
all_modules = list(sys.modules.values())
#with open('all_modules_loaded.txt', 'wt') as module_list_file:
with open(os.path.join (content_root,'all_modules_loaded.txt'), 'wt') as module_list_file:
    for module in all_modules:
        module_list_file.write(vars(module).get('__name__', ''))
        module_list_file.write('\n')

In [ ]:
for module in all_modules:
    __import__(module.__name__)

In [ ]:
from pathlib import PurePath

def local_file(module: dict) -> bool:
    v = vars(module)
    file = v.get('__file__')
    if file:
        try:
            return PurePath(file).relative_to(project_base)
        except ValueError:
            pass
        
custom = list(filter(local_file, sys.modules.values()))
print("Using {0} python files".format(len(custom)))

In [ ]:
custom_module_files = map(lambda f: str(f.__file__), custom)

In [ ]:
dependencies = set(custom_module_files)
len(dependencies)

In [ ]:
#vars(sys.modules['IM_DB.parameters'])

In [ ]:
vars(sys.modules['os.path'])

In [ ]:
backup_time = datetime.now().strftime("%Y-%m-%d")
#backup_file = "_publishable-" + data['model']['name'] + '-' + backup_time + '.zip'
backup_file = os.path.join(content_root,"_publishable-" + data['model']['name'] + '-' + backup_time + '.zip')
backup_file

In [ ]:
with zipfile.ZipFile(backup_file, 'w', compression=zipfile.ZIP_DEFLATED) as zipfile:
    zipfile.write(ssot)
    for dependency in dependencies:
        zipfile.write(dependency)
    print("Wrote {0}".format(backup_file))

In [ ]:
for diagram in data['diagrams'].values():
    relations = filter(lambda relation : len(relation['linesegments']) < 3 and 
                       len(set(map(lambda x: x['linetype'], relation['linesegments']))) < 2, diagram['relationships'].values())
    for r in relations:
        print(f"Found relation {r}")